In [1]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable,get_current_run_tree


In [2]:
from dotenv import load_dotenv
import os   

load_dotenv("../../.env")

True

embedding function

In [3]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"
        }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"]={
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.data[0].embedding   

Retrieval function

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

Format retrieved context function

In [6]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

Create prompt template function

In [7]:
@traceable(
    name="build_prompt",
    run_type="prompt"
)
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

Generate answer function

In [8]:
@traceable(
    name="generate_answer",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-nano",
        }
)
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"]={
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens":response.usage.completion_tokens,
            "total_tokens":response.usage.total_tokens,
        }
    return response.choices[0].message.content

Combined RAG pipeline

In [9]:
@traceable
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [10]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have the Hotsales Portable Neck Fan (B0C6JRWWFQ, 4.2 rating). It’s a hands-free, wearable 360° cooling neck fan with 4 speeds, quiet operation, and a rechargeable 5200 mAh battery (USB powered).


In [11]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Yes—here are earphones/headphones from the available products with a rating above 4:

1) Jesebang Wireless Earbud (Bluetooth Headphones 5.3, ENC Mic, USB-C, LED display, IP7 waterproof, sport earhooks, Black) — rating 4.9 (B0C2TLBSMP)

2) Cowyawn Toddler Headphones (ultra-light wired kids headphones, 85/94dB volume limit, built-in microphone, 3.5mm jack, navy/teal) — rating 4.5 (B0C2VVFQLD)

3) LUDOS Turbo Wired Earbuds with Microphone (in-ear, memory foam, durable cable, bass) — rating 4.4 (B07VSWK14G)

4) Earbay Bluetooth Headset with Microphone (ENC noise cancelling mic, USB dongle, mute button, multipoint, up to 26hrs talk time) — rating 4.0 (B0C6KM4H9M)


In [12]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

Sure. Here are the earphones/headphones in the available products list with a rating below 4.0:

1) LUDOS Turbo Wired Earbuds (B07VSWK14G) — rating 4.4 (Note: This is not below 4.0, so not included.)

2) Bluetooth Headset with Microphone + ENC (Earbay) (B0C6KM4H9M) — rating 4.0 (Note: This is exactly 4.0, so not below 4.0.)

3) None of the other earphone/earbud items shown have a rating below 4.0.

If you want, I can also suggest the ones below or equal to 4.5, or below or equal to 4.2.
